# 01. Preprocessing & Chunking Pipeline
This notebook cleans, normalizes, and chunks the ingested Ayurvedic data for use in RAG / Vector Search.

Steps applied:
1. Inspect Raw Data
2. Clean Data (drop nulls, trim whitespace, deduplicate)
3. Normalize Structure (add document ID)
4. Chunk text into overlapping windows
5. Add chunk index metadata
6. Save as Delta table

In [ ]:
# --- Configuration ---
CATALOG = "bricksiitm" 
SCHEMA = "ayurgenix"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.knowledge_base" 
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.processed_knowledge_base"

## Step 1: Inspect What Was Ingested

In [ ]:
print(f"🔍 Inspecting raw ingested data from {SOURCE_TABLE}...")
try:
    df = spark.table(SOURCE_TABLE)
    display(df.limit(10))
except Exception as e:
    print(f"⚠️ Error reading table: {e}\nDid you create the correct table in the previous step?")

## Step 2: Clean the Data
Remove nulls, trim whitespace, and drop duplicates to prevent injecting garbage data into LLMs.

In [ ]:
from pyspark.sql.functions import col, trim

print("🧹 Cleaning data...")
df_clean = (
    df
    .filter(col("text_chunk").isNotNull())
    .withColumn("text_chunk", trim(col("text_chunk")))
    .dropDuplicates()
)

print(f"Records after cleaning: {df_clean.count()}")

## Step 3: Normalize Structure
Add unique IDs to the rows so we can track the lineage of chunks back to their source document.

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id

print("🔢 Normalizing structure and adding document IDs...")

# Assign a unique Document ID to each original row.
df_clean = df_clean.withColumn("id", monotonically_increasing_id())

## Step 4: Text Chunking
Splitting the large text chunks into smaller, overlapping chunks suitable for ML embeddings and vector search.

In [ ]:
from pyspark.sql.functions import udf, explode
from pyspark.sql.types import ArrayType, StringType

def chunk_text(text, size=300, overlap=50):
    if not text:
        return []
    words = text.split()
    chunks = []
    
    # If the text is shorter than the size limit, keep it as a single chunk
    if len(words) <= size:
        return [" ".join(words)]
        
    for i in range(0, len(words), size - overlap):
        chunk = " ".join(words[i : i + size])
        chunks.append(chunk)
        
    return chunks

chunk_udf = udf(chunk_text, ArrayType(StringType()))

print("✂️ Chunking text...")

df_chunked = df_clean.withColumn("chunks", chunk_udf(col("text_chunk")))
df_final = df_chunked.select("*", explode(col("chunks")).alias("chunk_text"))

print(f"Total separate chunks generated: {df_final.count()}")

## Step 5: Add Chunk Metadata

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("🏷️ Adding chunk indices...")

window = Window.partitionBy("id").orderBy(col("id"))

df_final = df_final.withColumn(
    "chunk_id",
    row_number().over(window)
)

df_final = df_final.drop("text_chunk", "chunks")

## Step 6: Save Processed Data

In [ ]:
print(f"💾 Saving processed knowledge base to {TARGET_TABLE}...")

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE)

print("✅ Data successfully processed and chunked!")
display(spark.table(TARGET_TABLE).limit(10))